In [2]:
import pandas as pd
import sqlite3
import csv
import json

### Calculate the average signal from the timeseries

In [3]:
def get_avg_signal(time_series, fft_size):

    reshaped = time_series.reshape(-1, fft_size)
    return reshaped.mean(axis=0)

### Database

#### Save observation

In [1]:
def save_observation(db_filename,y_axis, url_drive, target,is_on, date_time, location,temperature, cloudy, rainy, windy, center_freq, receiver, RA, Dec, sample_rate, rf_gain, if_gain, bb_gain, integration_time,bandwidth,fft_size,fft_window,processing_mode ):
    """_summary_

    Args:
        db_filename (_type_): the database name
        y_axis (_type_): the spectrum values (Relative Power)
        url_drive (_type_): the url for the file in the drive
        target (_type_): target of the observation
        is_on (_type_): on observation then TRUE, else off observation FALSE
        date_time (_type_): _date and time
        location (_type_): _the location of observation
        temperature (_type_): the temperature
        cloudy (_type_): how cloudy the wheather was
        rainy (_type_): how rainy the wheather was
        windy (_type_): how windy the wheather was
        center_freq (_type_): the center frequency
        receiver (_type_): receiver
        RA (_type_): coordinates
        Dec (_type_): coordinates
        sample_rate (_type_): sample rate
        rf_gain (_type_): rf gain
        if_gain (_type_): if gain
        bb_gain (_type_): bb gain
        integration_time (_type_): integration time
        bandwidth (_type_): badnwidth
        fft_size (_type_): fft size
        fft_window (_type_): fft window
        processing_mode (_type_): processing mode
    """

   # Convert to json
    y_axis_json = json.dumps(y_axis.tolist())

    conn = sqlite3.connect(db_filename)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS spectrum_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            y_axis TEXT,
            url TEXT,
            target TEXT,
            is_on BOOLEAN,
            datetime TEXT,
            location TEXT,
            temperature REAL,
            cloudy TEXT,
            rainy TEXT,
            windy TEXT,
            center_freq REAL,
            receiver TEXT,
            RA TEXT,
            Dec TEXT,
            Sample_Rate REAL,
            RF_Gain INTEGER,
            IF_Gain INTEGER,
            BB_Gain INTEGER,
            integration_time REAL,
            Bandwidth REAL,
            FFT_Size INTEGER,
            FFT_Window TEXT,
            Proccesing_Mode TEXT   
            
        )
    ''')


    cursor.execute('''
        INSERT INTO spectrum_data (y_axis,url,target,is_on,datetime,location,temperature,cloudy,rainy,windy,center_freq,receiver,RA,Dec,Sample_Rate,RF_Gain,IF_Gain,BB_Gain,integration_time,Bandwidth,FFT_Size,FFT_Window,Proccesing_Mode)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (y_axis_json, url_drive, target,is_on, date_time, location,temperature, cloudy, rainy, windy, center_freq, receiver, RA, Dec, sample_rate, rf_gain, if_gain, bb_gain, integration_time,bandwidth,fft_size,fft_window,processing_mode))


    conn.commit()
    conn.close()
    
    print("Succesfuly saved")


#### Get data from the database

In [8]:
def get_data_from_query(database_filename, query):
    """
    Select rows from the database and return as dataframe using the query 

    Args:
        database_filename (_type_): the name of the database file
        query (_type_): the query to retrieve the data 

        e.g
        query = SELECT y_axis FROM spectrum_data WHERE target = "HI LINE"
    """

    conn = sqlite3.connect(database_filename)
    df = pd.read_sql_query(query,conn)
    conn.close()

    if 'y_axis' in df.columns:
        df['y_axis'] = df['y_axis'].apply(json.loads)
    
    return df


#### How to save an observation

In [ ]:


# The .csv file with the frequencies and relative power
observation_filename="2502202_Hot202020.csv"

# The fft size
fft_size = 2048

time_series_df = pd.read_csv(observation_filename)

# Keep only the relative power from the csv file
time_series_df = time_series_df.filter(regex='power_au|y_axis')

# Convert to numpy array
time_series_np = time_series_df.to_numpy()

# Create the averaged signal
average_signal = get_avg_signal(time_series_np,fft_size=fft_size)

# Save the observation to the database

# All params. The explanation is in the docstring

db_filename = "observations.db"
y_axis = average_signal
url_drive= " "
target = "HI LINE"
is_on = True
date_time = "Feb 25, 2026 6:58 PM"
location = "AUTH Observatory"
temperature = 11.0
cloudy = "Medium"
rainy = "Zero"
windy = "Medium"
center_freq = 1.42
receiver = "Δίπολο Υδρογόνου"
Ra = "06h 37, -s"
Dec = "+16° 23' 53.1\" "
sample_rate = 4
rf_gain = 6
if_gain = 20
bb_gain = 5
integration_time = 10
bandwidth = 3.84
fft_size = 2048
fft_window = "Hanning"
processing_mode = "On-Board(PC)"

save_observation(db_filename,y_axis,url_drive,target,is_on,date_time,location,temperature,cloudy,rainy,windy,center_freq,receiver,Ra,Dec,
                 sample_rate,rf_gain,if_gain,bb_gain,integration_time,bandwidth,fft_size,fft_window,processing_mode)


Succesfuly saved


#### How to retrieve data from the Database using SQLite query

In [ ]:
# Take the yaxis, the target name and the datetime from the observations where the temperature was lower than 16 degrees.
query = "SELECT y_axis,target,datetime FROM spectrum_data WHERE temperature < 16"

dataframe = get_data_from_query("observations.db",query)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   y_axis    1 non-null      object
 1   target    1 non-null      object
 2   datetime  1 non-null      object
dtypes: object(3)
memory usage: 156.0+ bytes
None
